# FiftyOne Demo: Image Classification with InceptionV3

This notebook demonstrates how to:

* Load the flowers classification dataset into FiftyOne
* Create splits for training, validation and testing
* Download the media to the FiftyOne Media Cache ONLY ONCE
* Export ONLY labels for the FiftyOne dataset, without exporting and duplicating
media unnecessarily
* Train an InceptionV3 model to classify over the 5 classes of flowers
* Save the model weights
* Apply the trained model on the test set of images that already exist in the 
FiftyOne Media Cache
* Write the resulting predictions to a manifest, then ingest those labels as
predictions back into the original dataset, without redundant export of media or
extra copying. 

In [4]:
import os

print(os.environ.get('FIFTYONE_API_URI'))

https://demo-api.fiftyone.ai


In [5]:
from pathlib import Path
import json
import time
from typing import List, Tuple

import fiftyone as fo
import fiftyone.utils.random as four

from PIL import Image
from tqdm import tqdm
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, models, transforms

# Note: Media Cache Size

Ensure that the FiftyOne Media Cache size is larger than the dataset size. Note
that the default is 32 GB. 

In [6]:
# ensure the media cache config is large enough to hold the whole dataset
fo.media_cache_config.cache_size_bytes=-1 #default is 32GB

In [7]:
DATASET_DIR='gs://voxel51-test/dwiref/flowers'
# Replace dataset directory with path to Azure where dataset is saved
# Alternatively, if the dataset is stored locally, replace with local path

dataset_name="flowers_classification_dataset"

# device="cuda" if torch.cuda.is_available() else "cpu"
device = torch.device("mps")

In [8]:
# Ingest dataset as type ImageClassificationDirectoryTree
try:
    dataset = fo.Dataset.from_dir(
        dataset_type=fo.types.ImageClassificationDirectoryTree,
        dataset_dir=DATASET_DIR,
        name=dataset_name,
        persistent=True,
        overwrite=True,
    )
except ValueError:
    print(f'A dataset with the name {dataset_name} already exists! \n',
          f'You can load the existing dataset instead of creating a new one or,',
          f'delete the existing dataset and try again.')

# If the dataset already exists, you can load it by uncommenting this line:
# dataset = fo.load_dataset(dataset_name)

# download media to media cache
dataset.download_media()

# compute metadata for performance
dataset.compute_metadata()

# Load classes and their count
flowers_classes = dataset.default_classes
num_classes = len(flowers_classes)
print(f'There are {num_classes} in this FiftyOne dataset. The classes are:',
      f'{flowers_classes}')

 100% |███████████████| 3170/3170 [2.8s elapsed, 0s remaining, 1.1K samples/s]      
Computing metadata...
 100% |███████████████| 3170/3170 [2.3s elapsed, 0s remaining, 1.4K samples/s]      
There are 5 in this FiftyOne dataset. The classes are: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


In [9]:
# Confirm media cache size, then download dataset media to local cache
print(f'FiftyOne Cloud Media Cache Size in Bytes:',
      f'{fo.media_cache_config.cache_size_bytes}')

# Note: this will only work on a cloud-backed dataset
# Skip this step if DATASET_DIR is a local path
dataset.download_media()

# Print local filepath for first sample in dataset after cache download
print(f'{dataset.first().local_path}')
# /Users/dwiref/fiftyone/__cache__/media/gcs/voxel51-test/

FiftyOne Cloud Media Cache Size in Bytes: -1
/Users/dwiref/fiftyone/__cache__/media/gcs/voxel51-test/dwiref/flowers/daisy/100080576_f52e8ee070_n.jpg


To avoid exporting the dataset separately for training, we will treat the local
media cache directory as the root dataset directory for model training.

In [10]:
# Create train/test/validation splits

four.random_split(dataset, {"train": 0.7, "test": 0.2, "val": 0.1})
print(dataset.count_sample_tags())

{'test': 634, 'val': 317, 'train': 2219}


In [11]:
# Replace with path to root directory of the dataset in local media cache

cache_dir = fo.media_cache_config.cache_dir

LOCAL_DATASET_DIR = Path(f'{cache_dir}/media/gcs/voxel51-test/dwiref/flowers')

# Parameters
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3

In [12]:
train = dataset.match_tags('train')
val = dataset.match_tags('val')
test = dataset.match_tags('test')

In [13]:
# Export a manifest of filepaths with export_media='manifest' to avoid
# exporting media and creating unwanted redundancy

# Replace the manifest pattern path with any local path where you want to
# save the exported labels. Note, this will NOT export or copy images or media
# but only the filenames and their ground truth labels

MANIFEST_PATTERN = f'/Users/dwiref/Downloads/flowers/'
train.export(
    export_dir=MANIFEST_PATTERN+'/train',
    dataset_type=fo.types.FiftyOneImageClassificationDataset,
    export_media='manifest',
    classes=dataset.default_classes,
    label_field='ground_truth',
    pretty_print=True
)

val.export(
    export_dir=MANIFEST_PATTERN+'/val',
    dataset_type=fo.types.FiftyOneImageClassificationDataset,
    export_media='manifest',
    classes=dataset.default_classes,
    label_field='ground_truth',
    pretty_print=True
)

test.export(
    export_dir=MANIFEST_PATTERN+'/test',
    dataset_type=fo.types.FiftyOneImageClassificationDataset,
    export_media='manifest',
    classes=dataset.default_classes,
    label_field='ground_truth',
    pretty_print=True
)

Directory '/Users/dwiref/Downloads/flowers//train' already exists; export will be merged with existing files
 100% |███████████████| 2219/2219 [1.4s elapsed, 0s remaining, 1.6K samples/s]        
Directory '/Users/dwiref/Downloads/flowers//val' already exists; export will be merged with existing files
 100% |█████████████████| 317/317 [707.4ms elapsed, 0s remaining, 448.1 samples/s]   
Directory '/Users/dwiref/Downloads/flowers//test' already exists; export will be merged with existing files
 100% |█████████████████| 634/634 [830.6ms elapsed, 0s remaining, 763.3 samples/s]   


In [14]:
LOCAL_DATASET_DIR = Path(f'{fo.media_cache_config.cache_dir}/media/gcs/voxel51-test/dwiref/flowers')

MANIFEST_PATH_TRAIN = MANIFEST_PATTERN+'/train/labels.json'
MANIFEST_PATH_VAL = MANIFEST_PATTERN+'/val/labels.json'
MANIFEST_PATH_TEST = MANIFEST_PATTERN+'/test/labels.json'

#AL Simplify these Python syntax

# 1) --- load the manifests ---
with open(MANIFEST_PATH_TRAIN, 'r') as file:
    manifest_train = json.load(file)

with open(MANIFEST_PATH_VAL, 'r') as file:
    manifest_val = json.load(file)

with open(MANIFEST_PATH_TEST, 'r') as file:
    manifest_test = json.load(file)

idx2class: List[str] = manifest_train["classes"]
class2idx = {c: i for i, c in enumerate(idx2class)}
# {'daisy': 0, 'dandelion': 1, 'roses': 2, 'sunflowers': 3, 'tulips': 4}

def idx_to_subfolder(idx: int) -> str:
    """Map label index to the sub-folder name (“sunflowers”, …)"""
    return idx2class[idx]

# 2) --- build list of (path, idx) for training and validation ---
# train
train_sample_images: List[Tuple[Path, int]] = []
for base_name, idx in manifest_train["labels"].items():
    path = Path(LOCAL_DATASET_DIR / idx_to_subfolder(idx) / f"{base_name}.jpg")
    train_sample_images.append((path, idx))

# val
val_sample_images: List[Tuple[Path, int]] = []
for base_name, idx in manifest_val["labels"].items():
    path = Path(LOCAL_DATASET_DIR / idx_to_subfolder(idx) / f"{base_name}.jpg")
    val_sample_images.append((path, idx))

# test - create a list of image paths to predict on
test_sample_images: List[Path] = []
for base_name, idx in manifest_test["labels"].items():
    path = Path(LOCAL_DATASET_DIR / idx_to_subfolder(idx) / f"{base_name}.jpg")
    test_sample_images.append(path)

# 3) --- tiny custom Dataset around a list ---
class FlowerDataset(Dataset):
    def __init__(self, items, transform=None):
        self.items = items
        self.transform = transform
    
    def __len__(self): return len(self.items)
    
    def __getitem__(self, i):
        img_path, label = self.items[i]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

In [15]:
# Augmentations & preprocessing recommended for InceptionV3
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(299),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

val_tf = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

In [16]:
train_ds = FlowerDataset(train_sample_images, train_tf)
val_ds = FlowerDataset(val_sample_images, val_tf)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

In [17]:
# Load InceptionV3 with default pretrained weights
model = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT)

# Replace output layer with 5-unit classifier
model.fc = nn.Linear(model.fc.in_features, len(idx2class))

# Include InceptionV3's auxiliary classifier
model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, len(idx2class))
model = model.to(device)
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.AdamW(model.parameters(), lr=LR)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [18]:
def unpack_inception(out):
    """
    Returns (logits, aux_logits_or_None) for both training and eval.
    """
    # in eval mode, use only only main head
    if isinstance(out, torch.Tensor):
        return out, None
    # in training mode, you get a tuple
    else:
        return out.logits, out.aux_logits

In [19]:
def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            # may be Tensor or tuple
            out = model(X)
            logits, aux = unpack_inception(out)
            loss = criterion(logits, y)
            # only when aux present
            if train and aux is not None:
                loss += 0.4 * criterion(aux, y)
            if train:
                loss.backward()
                optimizer.step()
        running_loss += loss.item() * X.size(0)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
    return running_loss / total, correct / total


In [20]:
for epoch in range(1, EPOCHS+1):
    t0 = time.perf_counter()
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False)
    scheduler.step()
    dt = time.perf_counter() - t0
    print(f"[{epoch:02d}/{EPOCHS}] "
          f"train loss={train_loss:.4f} acc={train_acc:.3f} | "
          f"val loss={val_loss:.4f} acc={val_acc:.3f} | "
          f"{dt:.1f}s")

[01/20] train loss=1.1941 acc=0.689 | val loss=0.7208 acc=0.779 | 29.8s
[02/20] train loss=0.9631 acc=0.752 | val loss=0.4991 acc=0.823 | 26.4s
[03/20] train loss=0.8559 acc=0.784 | val loss=0.4513 acc=0.861 | 26.4s
[04/20] train loss=0.7860 acc=0.795 | val loss=0.4481 acc=0.820 | 26.4s
[05/20] train loss=0.7007 acc=0.812 | val loss=0.3344 acc=0.874 | 26.4s
[06/20] train loss=0.6517 acc=0.836 | val loss=0.3206 acc=0.874 | 26.4s
[07/20] train loss=0.6158 acc=0.841 | val loss=0.3356 acc=0.880 | 26.7s
[08/20] train loss=0.5645 acc=0.856 | val loss=0.3160 acc=0.896 | 26.5s
[09/20] train loss=0.4994 acc=0.877 | val loss=0.2530 acc=0.912 | 26.6s
[10/20] train loss=0.4533 acc=0.889 | val loss=0.2482 acc=0.918 | 26.6s
[11/20] train loss=0.4393 acc=0.890 | val loss=0.2208 acc=0.918 | 26.7s
[12/20] train loss=0.3601 acc=0.908 | val loss=0.2165 acc=0.931 | 26.9s
[13/20] train loss=0.3850 acc=0.902 | val loss=0.2221 acc=0.918 | 27.2s
[14/20] train loss=0.3373 acc=0.916 | val loss=0.1790 acc=0.943 

In [21]:
# Save model weights
# Replace the output path for where you would like to save these weights
torch.save(
    {
        "model_state_dict": model.state_dict(), 
        "class_names": idx2class
    },
    "/Users/dwiref/Voxel51/inceptionv3_flowers.pt"
)

In [22]:
# Replace these paths accordingly
preds_manifest = "/Users/dwiref/Voxel51/predictions.json"
trained_weights = "/Users/dwiref/Voxel51/inceptionv3_flowers.pt"

In [23]:
# Load model checkpoint for trained weights
ckpt = torch.load(trained_weights, map_location=device)
pred_model = models.inception_v3()
pred_model.fc = torch.nn.Linear(
    pred_model.fc.in_features,
    len(idx2class)
)
pred_model.AuxLogits.fc = nn.Linear(pred_model.AuxLogits.fc.in_features, len(idx2class))
pred_model.load_state_dict(ckpt["model_state_dict"])
pred_model.to(device).eval()

# For embeddings, extract all but the final fully connected layer
embed_model = torch.nn.Sequential(*(list(pred_model.children())[:-1]))

/var/folders/sq/g4qx38cs4cx3pvk7w4jlnqb40000gn/T/ipykernel_25180/1134574900.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(trained_weights, map_locati

Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

In [24]:
# Transformation to apply before predictions
test_tf = transforms.Compose([
    transforms.Resize(320), transforms.CenterCrop(299),
    transforms.ToTensor(),  transforms.Normalize([0.5]*3, [0.5]*3),
])

In [25]:
# Run predictions on the list of images from the test split, and save
# the results to a manifest that can be ingested using the FiftyOne importer
# for dataset type FiftyOneImageClassificationDataset
labels = {}
for img_path in test_sample_images:
    img_path = Path(img_path)
    x = test_tf(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = pred_model(x)
        pred_idx = pred.argmax(1).item()
    # key = base filename (no .jpg)
    key = f"{DATASET_DIR}/{img_path.parent.name}/{img_path.name}"
    labels[key] = pred_idx

In [26]:
dataset = fo.load_dataset('flowers_classification_dataset')

In [27]:
manifest_like = {"classes": dataset.default_classes, "labels": labels}
with open(preds_manifest, "w") as f:
    json.dump(manifest_like, f, indent=4)

In [28]:
# Ingest predictions on the test set back into the original dataset
test_pred_dataset = fo.Dataset.from_dir(
    dataset_type=fo.types.FiftyOneImageClassificationDataset,
    labels_path=preds_manifest,
    name='test-flowers-preds',
    overwrite=True
)

# The above ingestion puts predictions into a 'ground_truth' field
# by default. Rename the field before merging back into the original dataset
# to avoid overwriting and polluting pre-existing ground truth, otherwise
# the resulting merge will be difficult to undo
test_pred_dataset.rename_sample_field('ground_truth','preds')

# Merge prediction results back into the original dataset
dataset.merge_samples(test_pred_dataset)

 100% |█████████████████| 634/634 [2.6s elapsed, 0s remaining, 245.3 samples/s] 


In [29]:
test_pred_dataset

Name:        test-flowers-preds
Media type:  image
Num samples: 634
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    preds:            fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)

In [ ]:
# Compute embeddings for all samples in the dataset, then compute_visualization
emb_model = models.inception_v3(weights=None, aux_logits=False)  # base arch
ckpt_path = "/Users/dwiref/Voxel51/inceptionv3_flowers.pt"         # <- your .pth file
state_dict = torch.load(ckpt_path, map_location="cpu")
emb_model.load_state_dict(state_dict, strict=False)  # strict=False skips aux heads
emb_model.fc = torch.nn.Identity()            # remove final classifier
emb_model.eval()

# create a manifest for the whole dataset to generate embeddings with
dataset.export(
    export_dir=MANIFEST_PATTERN+'/embed',
    dataset_type=fo.types.FiftyOneImageClassificationDataset,
    export_media='manifest',
    classes=dataset.default_classes,
    label_field='ground_truth',
    pretty_print=True
)

with open(Path(MANIFEST_PATTERN+'/embed/labels.json')) as f:
    full_manifest = json.load(f)

classes = full_manifest["classes"]
labels  = full_manifest["labels"]

img_paths = [
    LOCAL_DATASET_DIR / f"{classes[idx]}/{basename}.jpg"
    for basename, idx in labels.items()
]


# Compute embeddings on the list of images from the dataset manifest, and save
iv3_embeddings = []
with torch.no_grad():
    for p in img_paths:
        # print(p)
        img = Image.open(p).convert("RGB")
        x = test_tf(x).unsqueeze(0)
        emb = embed_model(x).squeeze().cpu().numpy()   # [2048]
        #iv3_embeddings.append(emb)


Directory '/Users/dwiref/Downloads/flowers//embed' already exists; export will be merged with existing files


/var/folders/sq/g4qx38cs4cx3pvk7w4jlnqb40000gn/T/ipykernel_25180/2262319317.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(ckpt_path, map_locati

 100% |███████████████| 3170/3170 [1.7s elapsed, 0s remaining, 1.9K samples/s]        


NotImplementedError: The operator 'aten::_upsample_bilinear2d_aa.out' is not currently implemented for the MPS device. If you want this op to be added in priority during the prototype phase of this feature, please comment on https://github.com/pytorch/pytorch/issues/77764. As a temporary fix, you can set the environment variable `PYTORCH_ENABLE_MPS_FALLBACK=1` to use the CPU as a fallback for this op. WARNING: this will be slower than running natively on MPS.